In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import os
import cv2
from tqdm import tqdm

# TensorFlow and Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import plot_model

# Additional utilities
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# GPU configuration
physical_devices = tf.config.experimental.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"GPU available: {physical_devices[0]}")
else:
    print("GPU not available, using CPU")

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


GPU not available, using CPU
Libraries imported successfully!
TensorFlow version: 2.19.0
Keras version: 3.9.2


In [2]:
# Transfer Learning Implementation Class

class TransferLearningExperiment:
    """
    Comprehensive transfer learning implementation with different strategies
    """
    
    def __init__(self):
        self.models = {}
        self.histories = {}
        self.results = {}
        
    def create_base_model(self, model_name='VGG16', input_shape=(224, 224, 3), 
                         include_top=False, weights='imagenet'):
        """
        Create pre-trained base model
        
        Args:
            model_name: 'VGG16', 'ResNet50', 'MobileNetV2'
            input_shape: Input image shape
            include_top: Whether to include final classification layers
            weights: Pre-trained weights to use
            
        Returns:
            Pre-trained model
        """
        if model_name == 'VGG16':
            base_model = VGG16(weights=weights, include_top=include_top, 
                              input_shape=input_shape)
        elif model_name == 'ResNet50':
            base_model = ResNet50(weights=weights, include_top=include_top, 
                                 input_shape=input_shape)
        elif model_name == 'MobileNetV2':
            base_model = MobileNetV2(weights=weights, include_top=include_top, 
                                    input_shape=input_shape)
        else:
            raise ValueError(f"Unsupported model: {model_name}")
            
        return base_model
    
    def create_transfer_model(self, base_model, num_classes, strategy='feature_extraction'):
        """
        Create transfer learning model with specified strategy
        
        Args:
            base_model: Pre-trained base model
            num_classes: Number of output classes
            strategy: 'feature_extraction', 'fine_tuning', 'full_fine_tuning'
            
        Returns:
            Transfer learning model
        """
        # Freeze base model layers based on strategy
        if strategy == 'feature_extraction':
            base_model.trainable = False
        elif strategy == 'fine_tuning':
            # Freeze early layers, unfreeze later layers
            base_model.trainable = True
            fine_tune_at = len(base_model.layers) // 2
            for layer in base_model.layers[:fine_tune_at]:
                layer.trainable = False
        elif strategy == 'full_fine_tuning':
            base_model.trainable = True
        
        # Build the model
        model = models.Sequential([
            base_model,
            layers.GlobalAveragePooling2D(),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            layers.Dense(512, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            layers.Dense(num_classes, activation='softmax')
        ])
        
        # Compile with appropriate learning rate
        if strategy == 'feature_extraction':
            learning_rate = 0.001
        else:
            learning_rate = 0.0001  # Lower learning rate for fine-tuning
            
        model.compile(
            optimizer=optimizers.Adam(learning_rate=learning_rate),
            loss='categorical_crossentropy',
            metrics=['accuracy', 'top_3_accuracy']
        )
        
        return model
    
    def train_model(self, model, train_data, val_data, model_name, 
                   epochs=20, batch_size=32):
        """
        Train transfer learning model
        
        Args:
            model: Compiled model
            train_data: Training data generator
            val_data: Validation data generator
            model_name: Name for saving model
            epochs: Number of training epochs
            batch_size: Batch size for training
            
        Returns:
            Training history
        """
        # Callbacks
        callbacks = [
            EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
            ModelCheckpoint(f'models/best_{model_name}.h5', monitor='val_accuracy', 
                          save_best_only=True, verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, 
                            min_lr=1e-7, verbose=1)
        ]
        
        # Train the model
        history = model.fit(
            train_data,
            epochs=epochs,
            validation_data=val_data,
            callbacks=callbacks,
            verbose=1
        )
        
        return history
    
    def evaluate_model(self, model, test_data, model_name):
        """
        Evaluate model performance
        
        Args:
            model: Trained model
            test_data: Test data generator
            model_name: Model name for results
            
        Returns:
            Evaluation results dictionary
        """
        # Evaluate on test set
        test_loss, test_accuracy, test_top3_accuracy = model.evaluate(test_data, verbose=0)
        
        # Get predictions for detailed analysis
        predictions = model.predict(test_data, verbose=0)
        y_pred = np.argmax(predictions, axis=1)
        y_true = test_data.classes
        
        results = {
            'test_loss': test_loss,
            'test_accuracy': test_accuracy,
            'test_top3_accuracy': test_top3_accuracy,
            'predictions': predictions,
            'y_pred': y_pred,
            'y_true': y_true
        }
        
        self.results[model_name] = results
        return results

# Initialize experiment class
experiment = TransferLearningExperiment()
print("Transfer Learning Experiment class initialized!")


Transfer Learning Experiment class initialized!
